In [ ]:
import zipfile
import shutil
from pathlib import Path
import csv
import json
import re

import numpy as np
from PIL import Image
import pytesseract
import cv2
import xml.etree.ElementTree as ET


# =====================================================================
# CONFIG
# =====================================================================

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

OCR_CONF_THRESHOLD = 60
MIN_CONTRAST_PASS = 4.5
MIN_TEXT_HEIGHT_PX = 14
BLUR_THRESHOLD = 80.0


# =====================================================================
# WCAG UTILITIES
# =====================================================================

def srgb_to_linear(c):
    if c <= 0.04045:
        return c / 12.92
    return ((c + 0.055) / 1.055) ** 2.4


def relative_luminance(rgb):
    r, g, b = [x / 255.0 for x in rgb]
    return (
        0.2126 * srgb_to_linear(r) +
        0.7152 * srgb_to_linear(g) +
        0.0722 * srgb_to_linear(b)
    )


def contrast_ratio(c1, c2):
    L1 = relative_luminance(c1)
    L2 = relative_luminance(c2)
    return (max(L1, L2) + 0.05) / (min(L1, L2) + 0.05)


# =====================================================================
# STEP 1: Extract WORD media images (word/media/*)
# =====================================================================

def extract_word_media(docx_path, out_dir):
    docx_path = Path(docx_path)
    media_dir = out_dir / "media"
    media_dir.mkdir(parents=True, exist_ok=True)

    extracted = []

    with zipfile.ZipFile(docx_path, "r") as z:
        for file in z.namelist():
            if file.startswith("word/media/"):
                filename = Path(file).name
                target = media_dir / filename

                with z.open(file) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)

                extracted.append(filename)

    return extracted, media_dir


# =====================================================================
# STEP 2: Extract ALL text nodes (captions may be anywhere)
# =====================================================================

def get_all_text_nodes(root, ns):
    """Return list of (text_content, element_reference)."""
    nodes = []
    for t in root.findall(".//w:t", ns):
        if t.text:
            nodes.append((t.text, t))
    return nodes


# =====================================================================
# STEP 3: Build CAPTION → FIGURE sequence
# =====================================================================

def get_figure_sequence(docx_path):
    """
    Return ordered list:
      { caption, kind, image_file OR embedding }
    """
    docx_path = Path(docx_path)
    figure_seq = []

    with zipfile.ZipFile(docx_path, "r") as z:
        ns = {
            "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main",
            "a": "http://schemas.openxmlformats.org/drawingml/2006/main",
            "pic": "http://schemas.openxmlformats.org/drawingml/2006/picture",
            "o": "urn:schemas-microsoft-com:office:office",
        }

        # rId → {target, type}
        rels = {}
        rel_xml = z.read("word/_rels/document.xml.rels")
        rel_root = ET.fromstring(rel_xml)
        rel_ns = {"": "http://schemas.openxmlformats.org/package/2006/relationships"}

        for r in rel_root.findall("Relationship", rel_ns):
            rels[r.attrib["Id"]] = {
                "target": r.attrib.get("Target", ""),
                "type": r.attrib.get("Type", "")
            }

        # Document XML
        doc_xml = z.read("word/document.xml")
        root = ET.fromstring(doc_xml)

        # Collect ALL text nodes (captions may be anywhere)
        all_text_nodes = get_all_text_nodes(root, ns)

        # --- CAPTION REGEX (extremely robust, for all Figure formats) ---
        fig_re = re.compile(
            r"Figure[\s\u00A0]+([A-Za-z0-9]+(?:[-\u2010\u2011\u2012\u2013\u2014\u2212]\d+|\.\d+)?)",
            re.IGNORECASE
        )

        # Collect captions in reading order
        captions = []
        for idx, (text, _) in enumerate(all_text_nodes):
            m = fig_re.search(text)
            if m:
                caption = ("Figure_" + m.group(1)).replace(" ", "_")
                captions.append((caption, idx))

        print(f"[INFO] Captions found: {len(captions)}")

        # ---- FIGURE ASSOCIATION ----
        for i, (caption, start_idx) in enumerate(captions):
            end_idx = captions[i+1][1] if i+1 < len(captions) else len(all_text_nodes)

            assigned = False

            # Look forward among the XML nodes to find associated drawing
            for j in range(start_idx, end_idx):
                _, text_elem = all_text_nodes[j]

                # Find ancestor paragraph of this text node
                p = text_elem
                for _ in range(6):
                    if p.tag.endswith("}p"):
                        break
                    p = p.getparent() if hasattr(p, "getparent") else None
                if p is None:
                    continue

                # Search for ANY a:blip under this paragraph
                blip = p.find(".//a:blip", ns)
                if blip is not None:
                    rid = blip.attrib.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}embed")
                    if rid and rid in rels:
                        target = rels[rid]["target"]
                        rtype = rels[rid]["type"]

                        # Word raster image
                        if target.startswith("media/") and rtype.endswith("/image"):
                            figure_seq.append({
                                "caption": caption,
                                "kind": "word_media",
                                "image_file": Path(target).name
                            })
                            assigned = True
                            break

                        # Excel embedding
                        if target.startswith("embeddings/") or rtype.endswith("/oleObject"):
                            figure_seq.append({
                                "caption": caption,
                                "kind": "excel_embedding",
                                "embedding": Path(target).name
                            })
                            assigned = True
                            break

                # OLEObject detection
                ole = p.find(".//o:OLEObject", ns)
                if ole is not None:
                    rid = ole.attrib.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id")
                    if rid and rid in rels:
                        target = rels[rid]["target"]
                        figure_seq.append({
                            "caption": caption,
                            "kind": "excel_embedding",
                            "embedding": Path(target).name
                        })
                        assigned = True
                        break

    print(f"[INFO] Total figure-caption associations: {len(figure_seq)}")
    return figure_seq


# =====================================================================
# STEP 4: ADA ANALYSIS
# =====================================================================

def analyze_image_for_ada(image_path):
    img_path = Path(image_path)
    img = Image.open(img_path).convert("RGB")
    arr = np.array(img)

    h, w, _ = arr.shape

    ocr = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)

    has_text = False
    min_contrast = None
    min_text_height = None

    for i in range(len(ocr["text"])):
        text = ocr["text"][i].strip()
        try:
            conf = int(ocr["conf"][i])
        except ValueError:
            continue
        if not text or conf < OCR_CONF_THRESHOLD:
            continue

        has_text = True
        x, y, wb, hb = (
            ocr["left"][i],
            ocr["top"][i],
            ocr["width"][i],
            ocr["height"][i],
        )

        if min_text_height is None or hb < min_text_height:
            min_text_height = hb

        cx = min(x + wb // 2, w - 1)
        cy = min(y + hb // 2, h - 1)
        fg = arr[cy, cx]

        by = max(y - 2, 0)
        bg = arr[by, cx]

        cr = contrast_ratio(fg, bg)
        if min_contrast is None or cr < min_contrast:
            min_contrast = cr

    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    blur_metric = cv2.Laplacian(gray, cv2.CV_64F).var()

    contrast_ok = (min_contrast is None) or (min_contrast >= MIN_CONTRAST_PASS)
    text_ok = (min_text_height is None) or (min_text_height >= MIN_TEXT_HEIGHT_PX)
    blur_ok = blur_metric >= BLUR_THRESHOLD

    return {
        "image_file": img_path.name,
        "has_text": has_text,
        "min_contrast": min_contrast,
        "min_text_height_px": min_text_height,
        "blur_metric": blur_metric,
        "contrast_ok": contrast_ok,
        "text_size_ok": text_ok,
        "blur_ok": blur_ok,
        "ada_pass": contrast_ok and text_ok and blur_ok,
    }


# =====================================================================
# STEP 5: REPORT WRITER
# =====================================================================

def make_json_safe(v):
    if isinstance(v, np.generic):
        return v.item()
    if isinstance(v, dict):
        return {k: make_json_safe(val) for k, val in v.items()}
    if isinstance(v, list):
        return [make_json_safe(x) for x in v]
    return v


def write_reports(out_dir, results):
    out_dir = Path(out_dir)

    fields = [
        "figure_label",
        "source_type",
        "image_file",
        "has_text",
        "min_contrast",
        "min_text_height_px",
        "blur_metric",
        "contrast_ok",
        "text_size_ok",
        "blur_ok",
        "ada_pass",
        "alt_text",
    ]

    # CSV
    csv_path = out_dir / "ada_report.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fields)
        w.writeheader()
        for r in results:
            row = {k: r.get(k) for k in fields}
            w.writerow(row)

    # JSON
    json_path = out_dir / "ada_report.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(make_json_safe(results), f, indent=2)

    # HTML
    html_path = out_dir / "ada_report.html"
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("<html><body><h1>ADA Report</h1>")
        f.write("<table border='1' cellpadding='4'>")
        f.write("<tr><th>Figure</th><th>Image</th><th>Contrast</th>"
                "<th>Text Size</th><th>Blur</th><th>ADA</th><th>Alt Text</th></tr>")

        for r in results:
            f.write("<tr>")
            f.write(f"<td>{r['figure_label']}</td>")
            f.write(f"<td><img src='{r['image_rel']}' style='max-width:300px;'><br>{r['image_file']}</td>")
            f.write(f"<td>{r['min_contrast']}</td>")
            f.write(f"<td>{r['min_text_height_px']}</td>")
            f.write(f"<td>{r['blur_metric']:.1f}</td>")
            f.write(f"<td>{'PASS' if r['ada_pass'] else 'FAIL'}</td>")
            f.write(f"<td>{r.get('alt_text','')}</td>")
            f.write("</tr>")

        f.write("</table></body></html>")

    print("Reports written:")
    print("  CSV:", csv_path)
    print("  JSON:", json_path)
    print("  HTML:", html_path)


# =====================================================================
# STEP 6: MAIN PIPELINE
# =====================================================================

def run_pipeline_for_docx(docx_path, output_root, excel_image_dir):
    docx_path = Path(docx_path)
    output_root = Path(output_root)
    excel_image_dir = Path(excel_image_dir)

    out_dir = output_root / docx_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n----- BUILDING FIGURE SEQUENCE -----")
    figure_seq = get_figure_sequence(docx_path)

    print("\n----- EXTRACTING WORD MEDIA IMAGES -----")
    word_images, media_dir = extract_word_media(docx_path, out_dir)

    print("\n----- MATCHING FIGURES → IMAGES -----")
    results = []

    # local excel image copy dir
    excel_local = out_dir / "excel"
    excel_local.mkdir(exist_ok=True)

    # load alt-text for word images
    alt_map = {}  # (optional, could re-add)

    # For each figure
    for fig in figure_seq:
        caption = fig["caption"]
        kind = fig["kind"]

        if kind == "word_media":
            img_name = fig["image_file"]
            img_path = media_dir / img_name

            ada = analyze_image_for_ada(img_path)
            ada["figure_label"] = caption
            ada["source_type"] = "word"
            ada["image_file"] = img_name
            ada["alt_text"] = alt_map.get(img_name, "")
            ada["image_rel"] = f"media/{img_name}"
            results.append(ada)

        else:  # excel embedding
            embed = fig["embedding"]  # e.g. Microsoft_Excel_Worksheet7.xlsx
            base = Path(embed).stem   # Microsoft_Excel_Worksheet7

            matches = list(excel_image_dir.glob(f"*{base}*.png"))
            if not matches:
                print(f"[WARN] No matching Excel PNG for {caption} using '{base}'")
                continue

            src = matches[0]
            dst = excel_local / src.name
            if not dst.exists():
                shutil.copy2(src, dst)

            ada = analyze_image_for_ada(dst)
            ada["figure_label"] = caption
            ada["source_type"] = "excel"
            ada["image_file"] = src.name
            ada["alt_text"] = ""
            ada["image_rel"] = f"excel/{src.name}"
            results.append(ada)

    print("\n----- WRITING REPORTS -----")
    write_reports(out_dir, results)

    print("\nPipeline complete.")
    print("Results stored in:", out_dir)


# =====================================================================
# MAIN
# =====================================================================

if __name__ == "__main__":
    DOCX_FILE = r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25.docx"
    OUTPUT_ROOT = r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\ada_output"
    EXCEL_IMAGE_DIR = r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\excel_images"

    run_pipeline_for_docx(DOCX_FILE, OUTPUT_ROOT, EXCEL_IMAGE_DIR)


Extracted: word/media/image1.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image1.png
Extracted: word/media/image2.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image2.png
Extracted: word/media/image3.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image3.png
Extracted: word/media/image4.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image4.png
Extracted: word/media/image5.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image5.png
Extracted: word/media/image6.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Repor

In [ ]:
import zipfile





















docx = r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25.docx"

with zipfile.ZipFile(docx, "r") as z:
    for name in z.namelist():
        print(name)

[Content_Types].xml
_rels/.rels
word/document.xml
word/drawings/drawing3.xml
word/drawings/drawing2.xml
word/drawings/drawing5.xml
word/drawings/drawing4.xml
word/_rels/document.xml.rels
word/drawings/drawing1.xml
word/footer2.xml
word/header2.xml
word/header1.xml
word/footer1.xml
word/_rels/footnotes.xml.rels
word/footnotes.xml
word/endnotes.xml
word/charts/_rels/chart14.xml.rels
word/charts/_rels/chart15.xml.rels
word/charts/_rels/chart16.xml.rels
word/charts/_rels/chart17.xml.rels
word/charts/_rels/chart18.xml.rels
word/charts/_rels/chart19.xml.rels
word/charts/_rels/chart20.xml.rels
word/charts/_rels/chart21.xml.rels
word/charts/_rels/chart22.xml.rels
word/media/image1.png
word/comments.xml
word/media/image2.png
word/media/image3.png
word/media/image4.png
word/charts/chart1.xml
word/charts/style1.xml
word/charts/colors1.xml
word/charts/chart2.xml
word/charts/style2.xml
word/charts/colors2.xml
word/charts/chart3.xml
word/charts/style3.xml
word/charts/colors3.xml
word/theme/themeOver